In [0]:
tabla_destino = "platform_dev.gold_byma.dim_fecha"

dbutils.widgets.text("full_load", "0")
full_load = int(dbutils.widgets.get("full_load"))
carga_full = (full_load == 1)

import logging
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("dimension_fecha")

In [0]:
try:
    if not spark.catalog.tableExists(tabla_destino):
        logger.warning(f"La tabla {tabla_destino} no existe. Creando...")
        dbutils.notebook.run("../DDL/creacion_tablas", 0)
        carga_full = True
    else:
        tiene_datos = spark.sql(f"SELECT 1 FROM {tabla_destino} LIMIT 1").take(1) != []
        if not tiene_datos:
            carga_full = True
except Exception:
    logger.exception(f"Error al validar {tabla_destino}")
    raise

In [0]:
logger.info("Generando serie temporal del calendario para el período 2026")

df_calendario = (
    spark.range(0, 120)
    .withColumn("fecha", F.expr("DATE_ADD('2026-01-01', cast(id as int))"))
    .withColumn("fecha_sk", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("dia_semana", F.date_format("fecha", "EEEE"))
    .withColumn("es_fin_de_semana", F.dayofweek("fecha").isin(1, 7))
    .withColumn("es_feriado_bursatil", F.lit(False))
    .select("fecha_sk", "fecha", "anio", "mes", "trimestre", "dia_semana", "es_fin_de_semana", "es_feriado_bursatil")
)

df_calendario.createOrReplaceTempView("v_calendario_preparado")

In [0]:
try:
    # Al no depender de identidad autogenerada en el motor, el OVERWRITE directo es seguro y eficiente
    if carga_full or spark.table(tabla_destino).count() == 0:
        logger.warning(f"Escribiendo calendario completo en {tabla_destino}")
        df_calendario.write.format("delta").mode("overwrite").saveAsTable(tabla_destino)
        logger.info("Calendario guardado con éxito.")
    else:
        logger.info("La dimensión de fechas ya se encuentra poblada. Omitiendo recarga.")
except Exception:
    logger.exception(f"Error al escribir en {tabla_destino}")
    raise